# ERP003950_fastp_q30_u40 — отчёт об аннотации мышинных последовательностей

Из шести итоговых таблиц IgBLAST AIRR формируется единый автономный HTML-отчёт. Сводная статистика охватывает все строки AIRR; `igblastr::igbrowser()` визуализирует по 5 репрезентативных продуктивных последовательностей из каждого образца.

In [ ]:
import os
from pathlib import Path
import subprocess, sys, time

_BCR_ENV = Path('/Users/epishkin/mamba/envs/bcr_env')
os.environ['PATH'] = str(_BCR_ENV / 'bin') + ':' + os.environ.get('PATH', '')

def find_repo(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'scripts' / 'airr_igbrowser_report.py').exists():
            return p
    raise FileNotFoundError('Cannot locate repository root')

REPO = find_repo()
DATASET_ROOT = REPO / 'results' / 'ERP003950'
REPORT_DIR = DATASET_ROOT / 'annotation' / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ENV = Path(sys.executable).resolve().parent.parent
PYTHON = ENV / 'bin' / 'python'
RSCRIPT = ENV / 'bin' / 'Rscript'
print('REPO=', REPO)
print('ENV=', ENV)
print('REPORT_DIR=', REPORT_DIR)
assert (DATASET_ROOT / 'annotation' / 'igblast').is_dir()

In [ ]:
def run_logged(name, cmd):
    log_path = REPORT_DIR / f'{name}.log'
    started = time.time()
    with log_path.open('w') as log:
        proc = subprocess.Popen([str(x) for x in cmd], cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True)
        print(f'[{name}] PID={proc.pid}; log={log_path}')
        sys.stdout.flush()
        while proc.poll() is None:
            elapsed = time.time() - started
            size = log_path.stat().st_size if log_path.exists() else 0
            print(f'[{name}] elapsed={elapsed/60:.1f} min; log={size/1e6:.2f} MB; PID={proc.pid}')
            sys.stdout.flush()
            time.sleep(30)
    if proc.returncode != 0:
        print(log_path.read_text()[-8000:])
        raise RuntimeError(f'{name} failed: rc={proc.returncode}')
    print(log_path.read_text()[-4000:])
    return log_path


In [ ]:
run_logged('annotation_report_stats', [
    PYTHON, REPO / 'scripts' / 'airr_igbrowser_report.py', 'stats',
    '--dataset-root', DATASET_ROOT,
])


In [ ]:
run_logged('annotation_report_igbrowser', [
    RSCRIPT, REPO / 'scripts' / 'render_igbrowser.R',
    REPORT_DIR / 'igbrowser_representatives.airr.tsv',
    REPORT_DIR / 'igbrowser_representatives_native.html',
])


In [ ]:
run_logged('annotation_report_assemble', [
    PYTHON, REPO / 'scripts' / 'airr_igbrowser_report.py', 'assemble',
    '--dataset-root', DATASET_ROOT,
])
FINAL = REPORT_DIR / 'ERP003950_annotation_igbrowser_report.html'
assert FINAL.exists() and FINAL.stat().st_size > 0
print('FINAL:', FINAL, FINAL.stat().st_size, 'bytes')
